# Working with Individual Sources

While `search_awards()` provides a unified interface, you can also query individual funding sources directly. Each source class exposes a `get_data()` method with source-specific parameters.

This notebook walks through several sources and their unique capabilities.

## NSF — National Science Foundation

The NSF source supports filtering by CFDA number (program code), which lets you target specific directorates like Biological Sciences or Computer Science.

In [ ]:
from award_pynder.sources.nsf import NSF, NSFPrograms, NSF_PROGRAM_TO_CFDA_NUMBER_LUT

# Show available NSF program codes
print("NSF Programs and their CFDA numbers:")
for prog, cfda in NSF_PROGRAM_TO_CFDA_NUMBER_LUT.items():
    print(f"  {prog:55s} -> {cfda}")

In [ ]:
# Search NSF Biological Sciences for a specific date range
bio_grants = NSF.get_data(
    query="genome",
    from_datetime="2023-01-01",
    to_datetime="2023-03-01",
    cfda_number=NSF_PROGRAM_TO_CFDA_NUMBER_LUT[NSFPrograms.Biological_Sciences],
    tqdm_kwargs={"leave": False},
)

print(f"Found {len(bio_grants)} Biological Sciences grants mentioning 'genome'")
bio_grants[["title", "pi", "institution", "amount"]].head(10)

## NIH — National Institutes of Health

The NIH source uses the NIH RePORTER API and supports full-text search of abstracts. Note the 10,000 grant hard limit — narrow your date range for broad queries.

In [ ]:
from award_pynder.sources.nih import NIH

nih_grants = NIH.get_data(
    query="single cell RNA sequencing",
    from_datetime="2023-01-01",
    to_datetime="2023-02-01",
    tqdm_kwargs={"leave": False},
)

print(f"Found {len(nih_grants)} NIH grants")
print(f"\nAgency codes: {nih_grants['program'].unique().tolist()}")
print(f"Total funding: ${nih_grants['amount'].sum():,.0f}")
nih_grants[["title", "pi", "institution", "amount"]].head(10)

## Mellon Foundation

The Mellon Foundation source uses a GraphQL API. It searches by keyword and year range, and fetches individual grant amounts in a second pass.

In [ ]:
from award_pynder.sources.mellon import Mellon

mellon_grants = Mellon.get_data(
    query="digital humanities",
    from_datetime="2022-01-01",
    to_datetime="2023-01-01",
    tqdm_kwargs={"leave": False},
)

print(f"Found {len(mellon_grants)} Mellon grants")
if not mellon_grants.empty:
    print(f"Grant-making areas: {mellon_grants['program'].dropna().unique().tolist()}")
    mellon_grants[["title", "institution", "amount", "year"]].head(10)

## Sloan Foundation

The Sloan source scrapes the grants database HTML. Date filtering is done client-side after fetching all matching grants.

In [ ]:
from award_pynder.sources.sloan import Sloan

sloan_grants = Sloan.get_data(
    query="data science",
    from_datetime="2022-01-01",
    to_datetime="2023-01-01",
    tqdm_kwargs={"leave": False},
)

print(f"Found {len(sloan_grants)} Sloan grants")
if not sloan_grants.empty:
    sloan_grants[["title", "institution", "pi", "amount", "year"]].head(10)

## Gates Foundation

The Gates Foundation source uses their JSON grants search API.

In [ ]:
from award_pynder.sources.gates import Gates

gates_grants = Gates.get_data(
    query="education",
    from_datetime="2022-01-01",
    to_datetime="2023-01-01",
)

print(f"Found {len(gates_grants)} Gates grants")
if not gates_grants.empty:
    gates_grants[["institution", "amount", "year"]].head(10)

## Standard Output Schema

Every source returns a DataFrame with the same 12 columns, making it easy to combine and compare results across funders:

In [ ]:
from award_pynder.sources.base import ALL_DATASET_FIELDS
import pandas as pd

schema = pd.DataFrame({
    "Field": ALL_DATASET_FIELDS,
    "Description": [
        "Grantee organization",
        "Principal investigator",
        "Award year",
        "Project start date",
        "Project end date",
        "Funding program or agency",
        "Award amount",
        "Unique grant identifier",
        "Grant title",
        "Grant abstract or description",
        "Search keyword used",
        "Data source name",
    ],
})
schema